<a href="https://colab.research.google.com/github/shareeep/11th-DAP-hate-speech/blob/shariff/hate-speech-llm-colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hate Speech Classification with LLMs (Colab Version)

This notebook explores using Large Language Models (LLMs) for hate speech classification on the HateXplain dataset.
We will use a prompt engineering approach to classify text into "hatespeech", "normal", or "offensive".

> **Important:** Please ensure you are using a **T4 GPU** runtime in Google Colab for faster inference.
> Go to **Runtime** > **Change runtime type** > Select **T4 GPU**.

In [1]:
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.6 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,201 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/u

In [2]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [3]:
!ollama pull gemma3:4b

In [4]:
!pip install langchain-ollama

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM
from IPython.display import Markdown

template = """Question: {question}

Answer: Let's think step by step."""

prompt = ChatPromptTemplate.from_template(template)

model = OllamaLLM(model="gemma3:4b")

chain = prompt | model

display(Markdown(chain.invoke({"question": "What's the length of hypotenuse in a right angled triangle"})))

Okay, that's a good start! "Let's think step by step" is a perfectly reasonable response when approaching a problem like this. 

Now, we need to continue the explanation. Here's a possible continuation of the answer:

"The length of the hypotenuse in a right-angled triangle can be found using the Pythagorean theorem. The Pythagorean theorem states that a² + b² = c², where 'a' and 'b' are the lengths of the two shorter sides (the legs) of the right triangle, and 'c' is the length of the hypotenuse (the side opposite the right angle).

For example, if one side is 3 and the other is 4, then 3² + 4² = c²  =>  9 + 16 = c²  =>  25 = c²  =>  c = √25 = 5.

Therefore, the length of the hypotenuse is √c² = c."

Do you want me to:

*   Expand on that explanation further?
*   Give you another example?
*   Explain the Pythagorean theorem in more detail?

In [ ]:
!pip install -q openai "datasets<3.0" "pyarrow<20" scikit-learn pandas python-dotenv tqdm

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Initialize OpenAI Client for Local Ollama
# Ollama provides an OpenAI-compatible API at /v1
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama" # API key is required but ignored by Ollama
)

print("Client initialized for local Ollama.")

In [ ]:
# Data Processing Helpers (Reused from BERT Notebook)

ID2NAME = {0: "hatespeech", 1: "normal", 2: "offensive"}
NAME2ID = {v:k for k,v in ID2NAME.items()}

def _coerce_annotators(ann):
    if ann is None: return []
    if isinstance(ann, str):
        try: ann = json.loads(ann)
        except: return []
    if isinstance(ann, list):
        if len(ann) == 0: return []
        if isinstance(ann[0], dict): return ann
        out = []
        for x in ann:
            try: out.append({"label": int(x), "target": []})
            except: pass
        return out
    if isinstance(ann, dict):
        labels = ann.get("label", [])
        targets = ann.get("target", [])
        n = len(labels)
        out = []
        for i in range(n):
            try: lab = int(labels[i])
            except: continue
            tgt = targets[i] if (isinstance(targets, list) and i < len(targets)) else []
            out.append({"label": lab, "target": tgt if isinstance(tgt, list) else [tgt]})
        return out
    return []

def _coerce_rationales(rats):
    if rats is None: return []
    if isinstance(rats, str):
        try: rats = json.loads(rats)
        except: return []
    if isinstance(rats, list):
        out = []
        for r in rats:
            if isinstance(r, list):
                out.append([int(v) if isinstance(v, (int, bool)) else 0 for v in r])
        return out
    return []

def _collect_targets(anns):
    seen, out = set(), []
    for a in anns:
        for t in a.get("target", []) or []:
            if t not in seen:
                seen.add(t); out.append(t)
    return out

def _majority_vote_label(anns):
    pri = [0, 2, 1]
    counts = {0:0, 1:0, 2:0}
    for a in anns:
        lab = int(a.get("label", 1))
        if lab in counts: counts[lab] += 1
    if counts[0] == counts[1] == counts[2] == 0:
        return 1
    return sorted(counts.items(), key=lambda kv: (-kv[1], pri.index(kv[0])))[0][0]

def _majority_rationale(rats, n_tokens):
    if not rats: return [0] * n_tokens
    thresh = (len(rats) + 1) // 2
    agg = [0] * n_tokens
    for r in rats:
        for i, v in enumerate(r[:n_tokens]):
            agg[i] += int(v)
    return [1 if c >= thresh else 0 for c in agg]

def _pick_text(ex):
    if "post_tokens" in ex and isinstance(ex["post_tokens"], list):
        return " ".join(ex["post_tokens"])
    if "text" in ex: return str(ex["text"])
    if "cleaned_text" in ex: return str(ex["cleaned_text"])
    for v in ex.values():
        if isinstance(v, list) and all(isinstance(t, str) for t in v):
            return " ".join(v)
    return ""

def normalize_example(e):
    tokens = e.get("post_tokens", None)
    text = _pick_text(e)
    anns = _coerce_annotators(e.get("annotators", None))
    rats = _coerce_rationales(e.get("rationales", None))
    label = _majority_vote_label(anns)
    rat_mask = _majority_rationale(rats, len(tokens) if isinstance(tokens, list) else len(text.split()))
    targets = _collect_targets(anns)
    return {
        "text_norm": text.strip(),
        "label_norm": int(label),
        "rationale_mask": rat_mask,
        "targets": targets,
    }

In [ ]:
# Load and Prepare Data
print("Loading HateXplain dataset...")
ds = load_dataset("Hate-speech-CNERG/hatexplain", trust_remote_code=True)

# Normalize
# Removed num_proc=2 to avoid pickling errors on Windows where helper functions aren't found in worker processes
ds = ds.map(normalize_example)

# Keep only necessary columns
to_keep = ["text_norm", "label_norm"]
drop_cols = [c for c in ds["train"].column_names if c not in to_keep]
ds = ds.remove_columns(drop_cols)

# Filter empty
ds = ds.filter(lambda t: len(t.strip()) > 0, input_columns=["text_norm"])

In [ ]:
import numpy as np

# Select Test Set
test_ds = ds["test"]
print(f"Full Test set size: {len(test_ds)}")
# Select Test Set
test_ds = ds["test"].select(range(100))
print(f"Selected Test set size: {len(test_ds)}")

In [ ]:
# Restart Ollama Server (Run this if you get Connection Errors)
import subprocess
import time

print("Stopping any existing Ollama processes...")
!pkill ollama
time.sleep(2)

print("Starting Ollama server...")
process = subprocess.Popen("ollama serve", shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(10) # Wait for server to start

print("Server restarted. Verifying...")
!ollama list

In [ ]:
def classify_text(text, model="gemma3:4b"):
    """
    Classifies text using Local Ollama API.
    Make sure you have pulled the model: `ollama pull phi3`
    """
    # Optimization: Truncate very long texts.
    # Hate speech is usually evident in the first few sentences.
    if len(text) > 1500:
        text = text[:1500]

    system_prompt = (
        "You are an expert content moderator. "
        "Classify the following text into one of these three categories: 'hatespeech', 'normal', 'offensive'.\n\n"
        "Definitions:\n"
        "- 'hatespeech': Content that promotes violence, discrimination, or disparagement against protected groups (race, religion, gender, etc.). It aims to incite hatred.\n"
        "- 'offensive': Content that is rude, crude, or disrespectful (e.g., profanity, insults) but does NOT incite hatred or violence against protected groups.\n"
        "- 'normal': Content that is neither hate speech nor offensive.\n\n"
        "Provide ONLY the category name as your response, in lowercase."
    )

    user_prompt = f"Text: {text}"

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0, # Deterministic
            max_tokens=10,
            timeout=60
        )
        return response.choices[0].message.content.strip().lower()
    except Exception as e:
        print(f"Error: {e}")
        return "error"

# Test the function
# Ensure you have the model running in Ollama
sample_text = test_ds[0]['text_norm']
print(f"Sample Text: {sample_text}")
print(f"Prediction: {classify_text(sample_text)}")

In [ ]:
# Run Inference on Test Set
predictions = []
ground_truth = []
texts = []

print("Running inference...")
for example in tqdm(test_ds):
    text = example['text_norm']
    label_id = example['label_norm']
    true_label = ID2NAME[label_id]

    pred_label = classify_text(text)

    # Handle potential API errors or unexpected output
    if pred_label not in NAME2ID:
        # Simple fallback or mapping if the LLM is chatty (though prompt says ONLY label)
        if "hate" in pred_label: pred_label = "hatespeech"
        elif "offen" in pred_label: pred_label = "offensive"
        elif "norm" in pred_label: pred_label = "normal"
        else: pred_label = "normal" # Default fallback

    predictions.append(pred_label)
    ground_truth.append(true_label)
    texts.append(text)

# Create a DataFrame for results
results_df = pd.DataFrame({
    "text": texts,
    "ground_truth": ground_truth,
    "prediction": predictions
})

results_df.head()

In [ ]:
# Display full results with complete text
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

print("Complete Results - Full Text View:\n")
print(results_df.to_string(index=True))

In [ ]:
# Evaluation
print("\nClassification Report:")
print(classification_report(ground_truth, predictions, target_names=["hatespeech", "normal", "offensive"], labels=["hatespeech", "normal", "offensive"]))

print("\nConfusion Matrix:")
print(confusion_matrix(ground_truth, predictions, labels=["hatespeech", "normal", "offensive"]))